# Create Tables

## crm_cust_info

In [0]:
drop table if exists workspace.silver.crm_cust_info;
create table workspace.silver.crm_cust_info (
    id long,
    key string,
    first_name string,
    last_name string,
    marital_status string,
    gender string,
    cst_create_date date,
    _dwh_loaded_at date
);

## crm_prd_info

# Load Data

## crm_cust_info

In [0]:
truncate table workspace.silver.crm_cust_info;
insert into workspace.silver.crm_cust_info
with flag_duplicates as (
    select 
        *,
        row_number() over (partition by cst_id order by cst_create_date desc) as flag_duplic
    from workspace.bronze.crm_cust_info
)
select
    cst_id as id,
    cst_key as key,
    coalesce(trim(cst_firstname), 'n/a') as first_name,
    coalesce(trim(cst_lastname), 'n/a') as last_name,
    case when upper(trim(cst_marital_status)) = 'S' then 'Single'
         when upper(trim(cst_marital_status)) = 'M' then 'Married'
         else 'n/a' end as marital_status,
    case when upper(trim(cst_gndr)) = 'F' then 'Female'
         when upper(trim(cst_gndr)) = 'M' then 'Male'
         else 'n/a' end as gender,
    coalesce(cst_create_date, '2000-01-01') as cst_create_date,
    getdate() as _dwh_loaded_at
from flag_duplicates where flag_duplic = 1;

## crm_prd_info

In [0]:
select
    